# Milan pilot · Seven UTC days in July 2022

Inspect the real ERA5-Land daily maximum 2 m air temperature for 15–21 July. The selected grid centre near central Milan is 45.5°N, 9.2°E. This pilot checks the input measurements; the summer heatwave analysis also needs the 1991–2020 reference period.

Download the pilot using the data guide first. This notebook reads that file from the local cache. Set `HEATWAVE_CACHE_DIR` to use a different cache.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
from heatwave.ingest.cache import cache_entry, cached_payload
from heatwave.ingest.provider import load_month
from heatwave.ingest.requests import DATASET_DOI, request_plan

cache = root / os.environ.get("HEATWAVE_CACHE_DIR", "data/raw/cds")
label, request = next(request_plan("pilot"))
payload = cached_payload(cache_entry(cache, label))
dates, lat, lon, tx_c = load_month(payload, request)

## Maximum temperature at the selected grid centre

Each date is a UTC day. Temperatures below are converted from Kelvin and rounded only for display.

In [ ]:
row = np.abs(lat - 45.5).argmin()
col = np.abs(lon - 9.2).argmin()
print(f"Grid centre {lat[row]:.1f}°N, {lon[col]:.1f}°E")
print("UTC date     Daily maximum (°C)")
for day, value in zip(dates.astype("datetime64[D]"), tx_c[:, row, col]):
    measurement = f"{value:.2f}" if np.isfinite(value) else "missing"
    print(f"{day}   {measurement:>8}")

## Daily maximum across the study rectangle

Colours show temperatures at the delivered 0.1° resolution. Each tile extends to the midpoints between adjacent centres. The ring marks the grid centre in the table.

In [ ]:
def edges(centres):
    centres = np.asarray(centres, dtype=float)
    mid = (centres[:-1] + centres[1:]) / 2
    return np.r_[2 * centres[0] - mid[0], mid, 2 * centres[-1] - mid[-1]]

day = np.datetime64("2022-07-15")
layer = tx_c[dates.astype("datetime64[D]") == day].squeeze(axis=0)
if not np.isfinite(layer).any():
    raise ValueError("The requested date has no observed temperature.")
west_east, south_north = edges(lon), edges(lat)
nodata = "#a2aab2"
palette = plt.get_cmap("inferno").copy()
palette.set_bad(nodata)
fig, ax = plt.subplots(figsize=(8.1, 6.9))
fig.subplots_adjust(left=0.09, right=0.92, bottom=0.19, top=0.84)
mesh = ax.pcolormesh(west_east, south_north, np.ma.masked_invalid(layer), cmap=palette,
                     shading="flat", rasterized=True)
ax.set(xlabel="Longitude (°E)", ylabel="Latitude (°N)",
       xlim=west_east[[0, -1]], ylim=south_north[[0, -1]],
       aspect=1 / np.cos(np.deg2rad(lat.mean())))
ax.ticklabel_format(useOffset=False)
ax.plot(lon[col], lat[row], marker="o", linestyle="none", markersize=8,
        markerfacecolor="none", markeredgecolor="#162835", markeredgewidth=2.0)
scale = fig.colorbar(mesh, ax=ax, fraction=0.043, pad=0.035)
scale.set_label("Daily maximum 2 m air temperature (°C)")
fig.text(0.095, 0.94, "Daily maximum temperature around Milan", fontsize=18, weight="bold")
fig.text(0.095, 0.891, f"15 July 2022 UTC   ·   0.1° grid   ·   observed range {np.nanmin(layer):.1f}–{np.nanmax(layer):.1f} °C",
         fontsize=11, color="#344956")
fig.legend(handles=[
    Line2D([], [], marker="o", linestyle="none", color="#162835", markerfacecolor="none",
           label=f"Reference cell ({lat[row]:.1f}°N, {lon[col]:.1f}°E)"),
    Patch(facecolor=nodata, label=f"Missing ({np.isnan(layer).sum()} of {layer.size} cells)"),
], loc="lower left", bbox_to_anchor=(0.082, 0.068), ncol=2, frameon=False)
fig.text(0.095, 0.047, f"Source: Copernicus Climate Change Service / ECMWF, ERA5-Land · DOI {DATASET_DOI}",
         fontsize=8.5, color="#46545e")
out = root / "runs" / "pilot" / "temperature_2022-07-15.png"
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=170, facecolor="white", bbox_inches="tight", pad_inches=0.22)
plt.show()